# Network Intrusion Detection System
### UNSW-NB15 Dataset — Binary Classification (Normal vs Attack)

**Objective:** Train a machine learning classifier on network traffic features to detect cyber-attacks in real time.  
**Dataset:** [UNSW-NB15](https://www.kaggle.com/datasets/mrwellsdavid/unsw-nb15) — 257 K flows, 45 features, 9 attack categories.  
**Pipeline:** Data → Clean → Encode → Scale → VarianceThreshold → LogReg + RandomForest → Save

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded OK')

## 1. Data Loading

If the real UNSW-NB15 CSV files are present in `../data/`, they are loaded.  
Otherwise **synthetic data** with the same schema is generated for a quick smoke test of the full pipeline.

To get the real dataset:
```
python ../data/download_data.py
```

In [ ]:
def generate_synthetic_data(n_samples=10_000):
    """Generate UNSW-NB15-shaped data when real CSVs are absent."""
    rng = np.random.default_rng(42)
    n   = n_samples

    proto_cats   = ['udp','tcp','arp','ospf','icmp','igmp','sctp','unas']
    service_cats = ['-','http','ftp','smtp','ssh','dns','ftp-data',
                    'irc','pop3','snmp','ssl','dhcp','radius']
    state_cats   = ['INT','FIN','CON','REQ','RST','URH','URN',
                    'PAR','ECO','ECR','TST','MAS','TXD','URP','no']
    attack_cats  = ['Generic','Exploits','Fuzzers','DoS',
                    'Reconnaissance','Analysis','Backdoor','Shellcode','Worms']
    ttl_vals     = [0,29,30,62,63,64,125,126,127,128,254,255]

    labels = rng.choice([0,1], n, p=[0.36,0.64])

    return pd.DataFrame({
        'id':               range(1, n+1),
        'dur':              rng.exponential(0.1, n),
        'proto':            rng.choice(proto_cats, n),
        'service':          rng.choice(service_cats, n),
        'state':            rng.choice(state_cats, n),
        'spkts':            rng.integers(1, 100, n),
        'dpkts':            rng.integers(0, 100, n),
        'sbytes':           rng.integers(100, 10_000, n),
        'dbytes':           rng.integers(0, 10_000, n),
        'rate':             rng.exponential(100_000, n),
        'sttl':             rng.choice(ttl_vals, n),
        'dttl':             rng.choice(ttl_vals, n),
        'sload':            rng.exponential(1e6, n),
        'dload':            rng.exponential(1e6, n),
        'sloss':            rng.integers(0, 10, n),
        'dloss':            rng.integers(0, 10, n),
        'sinpkt':           rng.exponential(0.01, n),
        'dinpkt':           rng.exponential(0.01, n),
        'sjit':             rng.exponential(0.001, n),
        'djit':             rng.exponential(0.001, n),
        'swin':             rng.choice([0,255,1024,2048,4096,8192,65535], n),
        'stcpb':            rng.integers(0, 2**31, n),
        'dtcpb':            rng.integers(0, 2**31, n),
        'dwin':             rng.choice([0,255,1024,2048,4096,8192,65535], n),
        'tcprtt':           rng.exponential(0.001, n),
        'synack':           rng.exponential(0.001, n),
        'ackdat':           rng.exponential(0.001, n),
        'smean':            rng.integers(28, 2000, n),
        'dmean':            rng.integers(0, 2000, n),
        'trans_depth':      rng.integers(0, 10, n),
        'response_body_len':rng.integers(0, 10_000, n),
        'ct_srv_src':       rng.integers(1, 20, n),
        'ct_state_ttl':     rng.integers(0, 6, n),
        'ct_dst_ltm':       rng.integers(1, 20, n),
        'ct_src_dport_ltm': rng.integers(1, 20, n),
        'ct_dst_sport_ltm': rng.integers(1, 20, n),
        'ct_dst_src_ltm':   rng.integers(1, 20, n),
        'is_ftp_login':     rng.choice([0,1], n, p=[0.98,0.02]),
        'ct_ftp_cmd':       rng.choice([0,1,2,3,4,5], n, p=[0.95,0.02,0.01,0.01,0.005,0.005]),
        'ct_flw_http_mthd': rng.choice([0,1,2,3,4], n, p=[0.9,0.05,0.03,0.01,0.01]),
        'ct_src_ltm':       rng.integers(1, 20, n),
        'ct_srv_dst':       rng.integers(1, 20, n),
        'is_sm_ips_ports':  rng.choice([0,1], n, p=[0.99,0.01]),
        'attack_cat':       np.where(labels==0,'Normal',rng.choice(attack_cats, n)),
        'label':            labels,
    })

In [ ]:
# Notebook lives in notebooks/ — data is one level up in data/
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
DATA_DIR     = os.path.join(NOTEBOOK_DIR, '..', 'data')
MODEL_DIR    = os.path.join(NOTEBOOK_DIR, '..', 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

train_csv = os.path.join(DATA_DIR, 'UNSW_NB15_training-set.csv')
test_csv  = os.path.join(DATA_DIR, 'UNSW_NB15_testing-set.csv')

if os.path.exists(train_csv) and os.path.exists(test_csv):
    print('Loading real UNSW-NB15 dataset ...')
    train = pd.read_csv(train_csv)
    test  = pd.read_csv(test_csv)
    df    = pd.concat([train, test]).reset_index(drop=True)
    print(f'  {len(df):,} records loaded.')
else:
    print('CSV files not found — using synthetic data for smoke testing.')
    print('Place UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv in data/ for real results.')
    df = generate_synthetic_data(10_000)
    print(f'  {len(df):,} synthetic records generated.')

## 2. Exploratory Data Analysis

In [ ]:
print(df.info())
print('\nFirst 5 rows:')
print(df.head())
print('\nLabel distribution:')
print(df['label'].value_counts())

## 3. Preprocessing

Steps:
1. Drop NaN rows and duplicates
2. Drop non-feature columns (`id`, `attack_cat`)
3. Label-encode categorical columns (`proto`, `service`, `state`)
4. Standardise with `StandardScaler`
5. Remove near-zero-variance features with `VarianceThreshold(0.01)`

In [ ]:
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
print(f'Shape after cleaning: {df.shape}')

In [ ]:
cols_to_drop = ['id', 'attack_cat']
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

In [ ]:
# Keep one encoder per column so the backend can reverse-map unseen values
le_encoders = {}
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    le_encoders[col] = le

print('Encoded columns:', categorical_cols)

In [ ]:
X = df.drop('label', axis=1)
y = df['label']
feature_names = list(X.columns)
print(f'Features: {len(feature_names)}  |  Samples: {len(y)}')

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
sel        = VarianceThreshold(threshold=0.01)
X_selected = sel.fit_transform(X_scaled)
selected_feature_names = [feature_names[i]
                           for i, keep in enumerate(sel.get_support()) if keep]
print(f'VarianceThreshold kept {X_selected.shape[1]} / {len(feature_names)} features')

### Correlation Heatmap (selected features sample)

In [ ]:
# Show first 15 selected features to keep heatmap readable
sample_df = pd.DataFrame(X_selected[:, :15], columns=selected_feature_names[:15])
plt.figure(figsize=(12, 10))
sns.heatmap(sample_df.corr(), cmap='coolwarm', annot=False)
plt.title('Feature Correlation Heatmap (first 15 selected features)')
plt.tight_layout()
plt.show()

## 4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## 5. Model Training

We train two classifiers:
- **Logistic Regression** — fast linear baseline
- **Random Forest** — ensemble method, typically outperforms LR on this dataset

In [ ]:
print('Training Logistic Regression ...')
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)
print(f'Logistic Regression Accuracy: {accuracy_score(y_test, lr_preds):.4f}')

In [ ]:
print('Training Random Forest ...')
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)
print(f'Random Forest Accuracy : {accuracy_score(y_test, rf_preds):.4f}')
print(f'Random Forest F1-Score : {f1_score(y_test, rf_preds):.4f}')

## 6. Evaluation

In [ ]:
print('=== Random Forest Classification Report ===')
print(classification_report(y_test, rf_preds, target_names=['Normal', 'Attack']))

In [ ]:
models = ['Logistic Regression', 'Random Forest']
scores = [accuracy_score(y_test, lr_preds), accuracy_score(y_test, rf_preds)]

plt.figure(figsize=(6, 4))
sns.barplot(x=models, y=scores)
plt.title('Model Accuracy Comparison')
plt.ylim(0, 1)
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()

In [ ]:
importances = rf_model.feature_importances_
top_idx     = np.argsort(importances)[-10:]
top_names   = [selected_feature_names[i] for i in top_idx]

plt.figure(figsize=(8, 5))
plt.barh(top_names, importances[top_idx])
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
cm = confusion_matrix(y_test, rf_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Attack'], yticklabels=['Normal','Attack'])
plt.title('Confusion Matrix — Random Forest')
plt.ylabel('True')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

## 7. Save Model Artifacts

All artifacts are saved to `../models/` so the FastAPI backend can load them.

| File | Contents |
|------|----------|
| `rf_model.pkl` | Trained Random Forest |
| `lr_model.pkl` | Trained Logistic Regression |
| `scaler.pkl` | Fitted StandardScaler |
| `feature_selector.pkl` | Fitted VarianceThreshold |
| `label_encoders.pkl` | Dict of LabelEncoders per categorical column |
| `feature_names.pkl` | List of feature names (after encoding, before scaling) |

In [ ]:
joblib.dump(rf_model,      os.path.join(MODEL_DIR, 'rf_model.pkl'))
joblib.dump(lr_model,      os.path.join(MODEL_DIR, 'lr_model.pkl'))
joblib.dump(scaler,        os.path.join(MODEL_DIR, 'scaler.pkl'))
joblib.dump(sel,           os.path.join(MODEL_DIR, 'feature_selector.pkl'))
joblib.dump(le_encoders,   os.path.join(MODEL_DIR, 'label_encoders.pkl'))
joblib.dump(feature_names, os.path.join(MODEL_DIR, 'feature_names.pkl'))

print(f'All artifacts saved to {os.path.abspath(MODEL_DIR)}')
print('\nSaved files:')
for f in os.listdir(MODEL_DIR):
    if f.endswith('.pkl'):
        size = os.path.getsize(os.path.join(MODEL_DIR, f))
        print(f'  {f:35s} {size/1024:.1f} KB')

## 8. Next Steps

1. **Start the API server** (from project root):
   ```bash
   uvicorn backend.app:app --reload
   ```
2. **Open the frontend:** `frontend/index.html` in a browser.
3. **Explore the auto-docs:** http://localhost:8000/docs
4. **Train with real data:** Download the dataset with `python data/download_data.py` then re-run this notebook.
5. **Tune hyperparameters:** Try `GridSearchCV` with `{'n_estimators': [100,200], 'max_depth': [10,20]}`.